# Diffusion 기반 손 관절 In-betweening (CondMDI 방식)

이 노트북은 팀의 SILK Transformer 백본을 **CondMDI**(Cohan et al., "Flexible Motion
In-betweening with Diffusion Models", SIGGRAPH 2024, arXiv:2405.11126,
공식 코드: github.com/setarehc/diffusion-motion-inbetweening) 방식의 diffusion
모델로 전환한 구현입니다.

## 왜 Diffusion인가
SILK(결정적 L1 회귀)는 시작·끝 컨텍스트가 비슷하면(예: 쥐었다 편 뒤 원래 모양으로 돌아옴)
"아무 일도 없었다"는 가장 안전한 답으로 수렴하는 경향이 있습니다(regression to the mean).
이건 문헌에서 "multimodal ambiguity"로 알려진 문제이며, CondMDI를 비롯한 diffusion 기반
in-betweening 연구들이 정확히 이 문제를 풀기 위해 나왔습니다 — 하나의 정답을 회귀하는 대신
가능한 중간 동작들의 분포에서 샘플링합니다.

## 팀 확정 사항 반영
- 데이터셋: **How2Sign만** (CSL-Daily 제외)
- 위치 인코딩: 팀이 검증한 **목표-상대(target-relative) 인코딩** 그대로 재사용
  (RMIB의 time-to-arrival 설계와 문헌적으로 일치함이 확인됨)
- **왼손 flip**: `diag(1,-1,-1)` conjugation, **행(row) 기반** 6D 컨벤션
  (SignSparK 공식 소스 `pose_datasets_lmdb.py`로 100% 확인됨)
- 평가: train(학습) / dev(개인 검증) / test(팀 최종 비교), L=[5,10,20,30]
- 지표: L2Q(부호보정 쿼터니언), L2P(실제 MANO로 근사, 팀원들의 손수 제작 템플릿보다 정밀),
  NPSS(FFT 기반)
- 전처리: 팀원 1이 만든 offset=5 stride 기반 인덱스(`train_index.npz` 등)를 그대로 사용.
  **단, 팀원 1의 인덱스 자체엔 flip이 적용된 실제 값이 없고 `hand` 플래그(0=오른손,
  1=왼손)만 있음 — 이 노트북이 실제 데이터 로딩 시점에 flip을 적용합니다.**
- 텍스트 조건화(translation/gloss): **보류** — 윈도우 단위로 텍스트 정렬이 안 맞고
  수어 전용 특화 위험이 있어 팀 논의로 제외.
- 다중 키프레임(segment 기반 보너스 조건): gap 안에 `segment` 상 수어 중간점이
  우연히 포함되면 그 프레임도 조건으로 노출 (CondMDI의 "무작위 개수 키프레임" 학습
  철학과 일치, 고정 3개가 아니라 "있으면 활용"하는 방식).

**공식 코드 확인 사항**: CondMDI는 MDM(Motion Diffusion Model) 위에 지어졌고, MDM 계열은
노이즈(ε)가 아니라 **x0(원본)를 직접 예측**하는 것이 특징입니다 — 이 노트북도 x0-prediction을
따릅니다. 학습 커맨드(`train.train_condmdi --keyframe_conditioned`)가 시사하듯, 핵심은
**"학습 때부터 마스킹된 조건(관측 프레임은 그대로, 나머지만 노이즈)"을 명시적으로 가르치는 것**
입니다 — 단순 추론시점 imputation은 논문에서 이미 성능이 떨어짐이 확인됐습니다.


## 1. 환경 설정

In [2]:

!pip install -q lmdb

import os, math, pickle, random, inspect, io, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

SEED = 42
NUM_WORKERS = max(1, min(os.cpu_count() - 2, 8))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


Mounted at /content/drive
device: cuda


## 2. chumpy/MANO 설정 (L2P 평가용)

In [3]:

!pip install chumpy -q

if not hasattr(np, "bool"): np.bool = bool
if not hasattr(np, "int"): np.int = int
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "object"): np.object = object
if not hasattr(np, "str"): np.str = str
if not hasattr(np, "complex"): np.complex = complex
if not hasattr(np, "unicode"): np.unicode = str
if not hasattr(inspect, "getargspec"): inspect.getargspec = inspect.getfullargspec

import chumpy
print("chumpy 로드 성공")

!pip install -q smplx
import smplx

MANO_MODEL_ROOT = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST"  # mano 폴더의 부모 경로

mano_right = smplx.create(
    model_path=MANO_MODEL_ROOT, model_type="mano", is_rhand=True,
    use_pca=False, flat_hand_mean=False, num_pca_comps=45,
).to(DEVICE)

print("MANO 로드 완료")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


/tmp/ipykernel_558/2763592422.py:6: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"): np.object = object
/tmp/ipykernel_558/2763592422.py:7: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "str"): np.str = str


chumpy 로드 성공
MANO 로드 완료


## 3. 설정값

관절/차원, context, 평가 T값, diffusion 스텝 수 등. 팀 확정 사항 그대로 반영.


In [4]:

N_JOINTS = 15          # How2Sign hand_pose (손목 제외)
POSE_DIM = N_JOINTS * 6  # 90

C_CONTEXT = 10
EVAL_T_VALUES = [5, 10, 20, 30]
TRAIN_T_RANGE = (5, 30)
MAX_SEQ_LEN = C_CONTEXT + max(TRAIN_T_RANGE[1], max(EVAL_T_VALUES)) + 1  # 41

D_MODEL = 1024
N_HEADS = 8
N_LAYERS = 6
D_FF = 4096
DROPOUT = 0.1

N_DIFFUSION_STEPS = 1000   # DDPM 표준 스텝 수
BATCH_SIZE = 64            # diffusion은 SILK보다 forward pass가 무거워서(매 스텝 노이즈 샘플링)
                            # 조금 더 작게 시작, OOM 나면 더 줄이기

INDEX_DIR = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST/SignSparK_index"  # 팀원1 인덱스 위치
DATA_ROOT = "/content/signspark_local_data"  # LMDB 로컬 경로 (기존과 동일 관례)


## 4. 왼손 flip + 6D 유틸리티

팀이 SignSparK 공식 소스(`pose_datasets_lmdb.py`)로 검증한 그대로: **행(row) 기반**
Gram-Schmidt 컨벤션, `diag(1,-1,-1)` conjugation.


In [5]:

_LEFT_HAND_FLIP_MASK = np.array([
    [ 1.0, -1.0, -1.0],
    [-1.0,  1.0,  1.0],
    [-1.0,  1.0,  1.0],
], dtype=np.float32)


def rotation_6d_to_matrix_np(d6):
    '''6D -> (...,3,3). 행(row) 기반 -- pose_datasets_lmdb.py와 동일 컨벤션.'''
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = a1 / np.linalg.norm(a1, axis=-1, keepdims=True)
    b2 = a2 - np.sum(b1 * a2, axis=-1, keepdims=True) * b1
    b2 = b2 / np.linalg.norm(b2, axis=-1, keepdims=True)
    b3 = np.cross(b1, b2, axis=-1)
    return np.stack((b1, b2, b3), axis=-2)


def matrix_to_rotation_6d_np(mats):
    '''(...,3,3) -> 6D, 앞 2개 행.'''
    batch_dim = mats.shape[:-2]
    return mats[..., :2, :].copy().reshape(*batch_dim, 6)


def flip_left_hand_features(left_feats, n_joints=N_JOINTS):
    '''왼손(SMPLX-left) -> 오른손(WiLoR) 컨벤션으로 변환. 공식 로직과 동일.'''
    left = left_feats.astype(np.float32, copy=False)
    T_len = left.shape[0]
    pose_6d = left[:, :n_joints * 6].reshape(T_len * n_joints, 6)
    mats = rotation_6d_to_matrix_np(pose_6d) * _LEFT_HAND_FLIP_MASK
    flipped = matrix_to_rotation_6d_np(mats).reshape(T_len, n_joints * 6)
    if left.shape[-1] > n_joints * 6:
        return np.concatenate([flipped, left[:, n_joints * 6:]], axis=-1)
    return flipped


# ---- torch 버전 (모델/평가에서 사용, row 기준, pytorch3d 표준과 검증 완료) ----
def rotation_6d_to_matrix(d6):
    a1, a2 = d6[..., 0:3], d6[..., 3:6]
    b1 = torch.nn.functional.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = torch.nn.functional.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-2)


def matrix_to_axis_angle(R):
    batch_shape = R.shape[:-2]
    R_flat = R.reshape(-1, 3, 3)
    cos_theta = ((R_flat[:, 0, 0] + R_flat[:, 1, 1] + R_flat[:, 2, 2]) - 1) / 2
    cos_theta = cos_theta.clamp(-1 + 1e-7, 1 - 1e-7)
    theta = torch.acos(cos_theta)
    axis = torch.stack([
        R_flat[:, 2, 1] - R_flat[:, 1, 2],
        R_flat[:, 0, 2] - R_flat[:, 2, 0],
        R_flat[:, 1, 0] - R_flat[:, 0, 1],
    ], dim=-1)
    denom = (2 * torch.sin(theta)).clamp(min=1e-7).unsqueeze(-1)
    axis = axis / denom
    return (axis * theta.unsqueeze(-1)).reshape(*batch_shape, 3)


def sixd_sequence_to_axis_angle(seq_6d):
    if isinstance(seq_6d, np.ndarray):
        seq_6d = torch.from_numpy(seq_6d).float()
    return matrix_to_axis_angle(rotation_6d_to_matrix(seq_6d))

@torch.no_grad()
def mano_forward(mano_layer, hand_pose_aa, device=DEVICE):
    T = hand_pose_aa.shape[0]
    global_orient = torch.zeros(T, 3, device=device)
    hand_pose = hand_pose_aa.reshape(T, -1).to(device)
    betas = torch.zeros(T, 10, device=device)
    output = mano_layer(global_orient=global_orient, hand_pose=hand_pose, betas=betas, return_verts=True)
    return output.joints  # (T, 16, 3)


## 5. 팀 위치 인코딩 (목표-상대) 그대로 재사용

In [6]:

class RelativePositionalEncoding(nn.Module):
    '''목표 키프레임을 기준(상대위치 0)으로 한 학습 가능한 위치 임베딩.
    팀이 검증한 설계 그대로 -- RMIB의 time-to-arrival과 문헌적으로 일치.'''
    def __init__(self, d_model, max_len=200):
        super().__init__()
        self.max_len = max_len
        self.pos_embedding = nn.Embedding(2 * max_len + 1, d_model)

    def forward(self, x, rel_pos):
        idx = (rel_pos + self.max_len).clamp(0, 2 * self.max_len)
        return x + self.pos_embedding(idx)


## 6. Diffusion 유틸리티 — 노이즈 스케줄, timestep 임베딩

**Timestep 임베딩이란**: 학습은 정답에 노이즈를 단계적으로(t=1..T) 섞어가는 과정을 배우고,
추론은 순수 노이즈에서 거꾸로(T..1) 걷어내며 복원합니다. "노이즈가 살짝 낀 상태"와
"거의 완전한 노이즈 상태"는 모델이 해야 할 일이 다르므로, **"지금 몇 번째 단계인지(t)"를
위치 인코딩과 같은 방식(sinusoidal)으로 벡터화해서 입력에 더해줍니다.**


In [7]:

def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

class TimestepEmbedding(nn.Module):
    '''diffusion timestep t -> d_model 차원 벡터 (sinusoidal, 위치 인코딩과 동일한 수식).'''
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.SiLU(), nn.Linear(d_model, d_model),
        )

    def forward(self, t):
        half = self.d_model // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, d_model)
        return self.mlp(emb)


## 7. 모델 — SILK 백본 + timestep 조건 + x0 예측

MDM/CondMDI 계열은 노이즈(ε)가 아니라 **x0(원본 회전값)을 직접 예측**합니다. 입력은
`[노이즈 낀 회전(90) ; 관측여부 플래그(1)]`이고, 관측된(컨텍스트+목표+보너스 키프레임)
구간은 노이즈를 안 섞고 **깨끗한 정답을 그대로** 넣습니다(CondMDI의 핵심 - "학습 때부터
inpainting 패턴을 가르친다").


In [8]:

class DiffusionSILKHand(nn.Module):
    def __init__(self, pose_dim=POSE_DIM, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT, max_len=MAX_SEQ_LEN):
        super().__init__()
        self.input_proj = nn.Linear(pose_dim + 1, d_model)  # +1: 관측여부 플래그
        self.pos_enc = RelativePositionalEncoding(d_model, max_len=max_len)
        self.time_emb = TimestepEmbedding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, pose_dim)

    def forward(self, x_noisy_with_flag, rel_pos, t, valid_mask=None):
        '''x_noisy_with_flag: (B, T, pose_dim+1) -- 관측 구간은 이미 깨끗한 값으로 치환됨
        rel_pos: (B, T) 목표-상대 위치
        t: (B,) diffusion timestep
        '''
        h = self.input_proj(x_noisy_with_flag)
        h = self.pos_enc(h, rel_pos)
        h = h + self.time_emb(t).unsqueeze(1)  # 모든 프레임에 동일한 timestep 정보 브로드캐스트
        key_padding_mask = ~valid_mask if valid_mask is not None else None
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        return self.output_proj(h)  # x0 예측 (B, T, pose_dim)


In [9]:

import os, shutil

DRIVE_BACKUP = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST/SignSparK_lmdb_backup"


def setup_lmdb():
    os.makedirs(DATA_ROOT, exist_ok=True)
    drive_lmdb_dir = f"{DRIVE_BACKUP}/lmdb"
    local_lmdb_dir = f"{DATA_ROOT}/lmdb"

    def _all_present(base_dir):
        return all(
            os.path.exists(f"{base_dir}/{split}/How2Sign_reopt_{split}.lmdb/data.mdb")
            for split in ["train", "dev", "test"]
        )

    if _all_present(local_lmdb_dir):
        print("로컬에 이미 LMDB 있음, 그대로 사용")
        return

    if _all_present(drive_lmdb_dir):
        print("Drive 백업에서 로컬로 복사 중...")
        shutil.copytree(drive_lmdb_dir, local_lmdb_dir, dirs_exist_ok=True)
        print("복사 완료")
        return

    print("Drive 백업도 없음 -- 처음부터 다운로드 (한 번만, 이후엔 Drive 백업으로 재사용)")
    os.environ["HF_HUB_DISABLE_XET"] = "1"  # xet 백엔드 불안정 이력이 있어 비활성화
    if not os.path.exists("/content/SignSparK_repo"):
        os.system("git clone -q https://github.com/JianHe0628/SignSparK.git /content/SignSparK_repo")
    os.system(f'cd /content/SignSparK_repo && python tools/download_data.py '
              f'--datasets How2Sign --dest "{DATA_ROOT}"')

    print("Drive에 백업 중 (다음번엔 이 다운로드 단계를 생략할 수 있음)...")
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    shutil.copytree(local_lmdb_dir, drive_lmdb_dir, dirs_exist_ok=True)
    print("백업 완료")


setup_lmdb()

for split in ["train", "dev", "test"]:
    path = f"{DATA_ROOT}/lmdb/{split}/How2Sign_reopt_{split}.lmdb/data.mdb"
    exists = os.path.exists(path)
    size = f"({os.path.getsize(path)/1e9:.2f} GB)" if exists else ""
    print(f"{split}: {exists} {size}")

Drive 백업에서 로컬로 복사 중...
복사 완료
train: True (7.90 GB)
dev: True (0.45 GB)
test: True (0.59 GB)


## 8. 데이터 로더 — 팀원 1 인덱스 + 실제 LMDB + flip + segment

팀원 1의 `{split}_index.npz`(clip_idx, hand, start, [T])를 읽어 실제 프레임을 LMDB에서
가져오고, `hand==1`(왼손)이면 flip을 적용합니다. `segment`도 같이 가져와서, gap 구간 안에
수어 중간점이 있으면 보너스 키프레임으로 노출합니다.


In [10]:

import lmdb

def open_lmdb(split):
    path = f"{DATA_ROOT}/lmdb/{split}/How2Sign_reopt_{split}.lmdb"
    env = lmdb.open(path, readonly=True, lock=False, readahead=False, meminit=False, max_readers=1024)
    return env


def load_clip_raw(env, clip_id):
    with env.begin() as txn:
        raw = txn.get(clip_id.encode() if isinstance(clip_id, str) else clip_id)
    npz = np.load(io.BytesIO(raw), allow_pickle=True)
    return {
        "segment": npz["segment"],
        "left_features": npz["left_features"][:, :POSE_DIM],
        "right_features": npz["right_features"][:, :POSE_DIM],
    }

def get_segment_bonus_frames(segment_arr):
    """공식 _segment_bounds 로직 이식. 각 수어 세그먼트의 (start, middle, end) 반환."""
    padded = [0] + list(segment_arr) + [0]
    bounds, start = [], None
    for i in range(1, len(padded)):
        cur, prev = padded[i], padded[i - 1]
        if cur in (1, 2) and prev == 0:
            start = i - 1
        elif cur == 0 and prev in (1, 2) and start is not None:
            end = i - 2
            bounds.append((start, (start + end) // 2, end))
            start = None
    return bounds


class DiffusionSignSparkDataset(Dataset):
    '''팀원1 인덱스 파일 기반. mode='train'이면 T를 매번 새로 뽑고(인덱스엔 start만 저장돼
    있음, SILK 논문 방식 그대로), mode='eval'이면 인덱스에 저장된 고정 T를 그대로 씀.'''

    def __init__(self, split, index_dir=INDEX_DIR, mode="train", seed=SEED):
        self.split = split
        self.mode = mode
        self.rng = np.random.default_rng(seed)

        idx_path = f"{index_dir}/{split}_index.npz"
        data = np.load(idx_path, allow_pickle=True)
        self.clip_ids = data["clip_ids"]
        self.clip_idx = data["clip_idx"]
        self.hand = data["hand"]
        self.start = data["start"]
        self.T_arr = data["T"] if "T" in data.files else None

        self.env = None  # lazy open (워커별로)
        self._clip_cache = {}  # 간단한 LRU 대용 -- 같은 클립이 여러 윈도우에 쓰이는 경우 재사용

    def _get_env(self):
        if self.env is None:
            self.env = open_lmdb(self.split)
        return self.env

    def _get_clip(self, clip_i):
        if clip_i not in self._clip_cache:
            if len(self._clip_cache) > 5000:  # 캐시 크기 제한
                self._clip_cache.clear()
            cid = self.clip_ids[clip_i]
            self._clip_cache[clip_i] = load_clip_raw(self._get_env(), cid)
        return self._clip_cache[clip_i]

    def __len__(self):
        return len(self.clip_idx)

    def __getitem__(self, i):
        clip_i = int(self.clip_idx[i])
        hand_flag = int(self.hand[i])
        start = int(self.start[i])
        clip = self._get_clip(clip_i)

        key = "left_features" if hand_flag == 1 else "right_features"
        feats = clip[key]
        if hand_flag == 1:
            feats = flip_left_hand_features(feats)

        if self.mode == "train":
            T = int(self.rng.integers(TRAIN_T_RANGE[0], TRAIN_T_RANGE[1] + 1))
        else:
            T = int(self.T_arr[i])

        L = C_CONTEXT + T + 1
        window = feats[start:start + L].astype(np.float32)          # (L, 90)
        segment_window = clip["segment"][start:start + L]            # (L,)

        target_rot = window.copy()
        obs_mask = np.zeros(L, dtype=bool)
        obs_mask[:C_CONTEXT] = True
        obs_mask[-1] = True

        # 보너스 키프레임 -- 학습 때만! (mode 체크 추가)
        if self.mode == "train":
            bonus_prob = 0.5  # CondMDI 식으로 항상 노출 대신 확률적으로 (다양성 확보)
            if self.rng.random() < bonus_prob:
                bounds = get_segment_bonus_frames(segment_window)
                for seg_start, mid, seg_end in bounds:   # start -> seg_start, end -> seg_end로 이름 변경
                    for frame_idx in (seg_start, mid, seg_end):
                        if C_CONTEXT <= frame_idx < L - 1:
                            obs_mask[frame_idx] = True

        rel_pos = np.arange(L, dtype=np.int64) - (L - 1)  # 목표(마지막 프레임) 기준 상대위치

        return {
            "target": torch.from_numpy(target_rot),     # (L, 90) 정답
            "obs_mask": torch.from_numpy(obs_mask),      # (L,) True=관측(컨텍스트/목표/보너스)
            "rel_pos": torch.from_numpy(rel_pos),
        }


def collate_diffusion(batch):
    L_max = max(b["target"].shape[0] for b in batch)
    B = len(batch)
    target = torch.zeros(B, L_max, POSE_DIM)
    obs_mask = torch.zeros(B, L_max, dtype=torch.bool)
    valid_mask = torch.zeros(B, L_max, dtype=torch.bool)
    rel_pos = torch.full((B, L_max), -9999, dtype=torch.int64)

    for i, b in enumerate(batch):
        L = b["target"].shape[0]
        target[i, :L] = b["target"]
        obs_mask[i, :L] = b["obs_mask"]
        valid_mask[i, :L] = True
        rel_pos[i, :L] = b["rel_pos"]

    return {"target": target, "obs_mask": obs_mask, "valid_mask": valid_mask, "rel_pos": rel_pos}


## 9. 학습 — CondMDI 스타일 마스킹 + x0 예측 손실

매 스텝: 정답에 노이즈를 섞되, **관측 구간(컨텍스트/목표/보너스 키프레임)은 노이즈를
안 섞고 깨끗한 값 그대로 유지**합니다. 모델은 전체 시퀀스에 대해 x0를 예측하고, 손실은
SILK와 마찬가지로 **전체 시퀀스**에 대해 계산합니다(팀이 gap-only 손실 ablation은
이번 스코프에서 보류하기로 했으므로 일관성 유지).


In [11]:

def training_step_flow(model, batch, device=DEVICE, use_amp=True):
    """DDPM 대신 Flow Matching. 노이즈(x0)에서 데이터(x1)로 가는 직선 경로의
    속도장(velocity = x1 - x0)을 예측하도록 학습. schedule 객체 불필요."""
    target = batch["target"].to(device)         # x1 (데이터)
    obs_mask = batch["obs_mask"].to(device)
    valid_mask = batch["valid_mask"].to(device)
    rel_pos = batch["rel_pos"].to(device)

    B, L, _ = target.shape
    noise = torch.randn_like(target)              # x0
    t = torch.rand(B, device=device)                # 연속시간 [0,1) -- 이산 스텝 아님

    t_expand = t.view(-1, 1, 1)
    x_t = (1 - t_expand) * noise + t_expand * target   # 직선 보간 경로
    true_velocity = target - noise                       # 학습 목표(속도장)

    # 관측 구간은 노이즈 없이 항상 정답 그대로 유지 (CondMDI의 inpainting 메커니즘 그대로)
    obs_mask_f = obs_mask.unsqueeze(-1).float()
    x_input = x_t * (1 - obs_mask_f) + target * obs_mask_f

    flag = obs_mask.float().unsqueeze(-1)
    model_in = torch.cat([x_input, flag], dim=-1)

    if use_amp and device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred_velocity = model(model_in, rel_pos, t, valid_mask)
    else:
        pred_velocity = model(model_in, rel_pos, t, valid_mask)

    diff = torch.abs(pred_velocity - true_velocity).mean(dim=-1)
    diff = diff * valid_mask.float()
    loss = diff.sum() / valid_mask.float().sum().clamp(min=1.0)
    return loss


def train_diffusion_silk(steps=127560, batch_size=64, lr=3e-4, weight_decay=1e-4,
                          save_dir="/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST",
                          log_every=2000, dev_eval_every=10000, dev_eval_n=1500,
                          save_every=10000, resume=True):
    """팀원 1의 exp06(train_offset5.py) 학습 프로토콜을 최대한 그대로 반영:
    - steps=127560, batch_size=64, lr=3e-4, weight_decay=1e-4 (팀원1과 동일)
    - OneCycleLR(max_lr=lr, total_steps=steps) (팀원1과 동일 스케줄)
    - epoch 기반이 아니라 step 고정 루프 -- train_loader를 무한 반복(팀원1과 동일 방식,
      DataLoader가 바닥나면 StopIteration을 잡아서 새로 순회 시작)
    - grad_norm clip max_norm=1.0, drop_last=True (동일)

    의도적으로 다르게 유지한 부분:
    - 팀원1은 loss가 NaN이면 학습 자체를 완전히 중단하지만, 우리는 그 배치만 스킵하고
      계속 진행 (Noam 스케줄러 버그를 겪으며 이 방식이 안전하다는 걸 이미 확인했음)
    - 팀원1은 최종 스텝의 가중치만 저장하지만, 우리는 주기적 dev 평가로 best 체크포인트도
      별도 추적 (같은 step 예산 안에서의 선택이라 step 수 비교의 공정성에는 영향 없음)
    """
    train_ds = DiffusionSignSparkDataset("train", mode="train")
    dev_ds = DiffusionSignSparkDataset("dev", mode="eval")

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_diffusion,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
        prefetch_factor=4, drop_last=True,   # 팀원1도 drop_last=True
    )

    model = DiffusionSILKHand().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=steps)

    latest_path = f"{save_dir}/diffusion_silk_flow_matched_latest.pt"
    best_path = f"{save_dir}/diffusion_silk_flow_matched_best.pt"

    start_step, best_val = 0, float("inf")

    if resume and os.path.exists(latest_path):
        ckpt = torch.load(latest_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_step = ckpt["step"]
        best_val = ckpt["best_val"]
        print(f"체크포인트에서 이어받음: step {start_step}부터, best_val={best_val:.4f}")

    model.train()
    it = iter(train_loader)
    losses = []
    t0 = time.time()

    for step in range(start_step + 1, steps + 1):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(train_loader)
            batch = next(it)

        optimizer.zero_grad()
        loss = training_step_flow(model, batch)

        if not torch.isfinite(loss):
            print(f"[경고] step {step}: loss가 NaN/Inf ({loss.item()}) -- 배치 스킵")
            continue

        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

        if step % log_every == 0 or step == 1:
            recent = np.mean(losses[-log_every:])
            print(f"step {step:7d}/{steps}  loss={recent:.4f}  grad_norm={gnorm:.3f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")

        if step % dev_eval_every == 0:
            model.eval()
            dev_loss = eval_dev_flow(model, dev_ds, n=dev_eval_n)
            model.train()
            print(f"   -> [dev, step {step}] loss: {dev_loss:.4f}")

            if dev_loss < best_val:
                best_val = dev_loss
                torch.save({"model_state": model.state_dict(), "step": step, "best_val": best_val},
                           best_path)
                print(f"   -> best 갱신, 저장됨")

        if step % save_every == 0 or step == steps:
            torch.save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(), "step": step, "best_val": best_val},
                       latest_path)

    dt = time.time() - t0
    print(f"\n학습 시간: {dt:.1f}s ({dt/max(steps,1)*1000:.1f}ms/step)")

    if os.path.exists(best_path):
        best_ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(best_ckpt["model_state"])
        print(f"best 체크포인트(step {best_ckpt['step']})로 복원")
    return model


def eval_dev_flow(model, dev_ds, n=1500, batch_size=128, seed=0):
    """팀원1의 eval_dev와 동일한 방식(무작위 표본 n개)으로 dev loss 확인."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(dev_ds), min(n, len(dev_ds)), replace=False)
    sub = torch.utils.data.Subset(dev_ds, idx)
    loader = DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=collate_diffusion)
    total, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            total += training_step_flow(model, batch).item()
            n_batches += 1
    return total / max(n_batches, 1)


## 10. 추론(샘플링) — DDPM 역과정, 매 스텝 관측 구간 재치환

CondMDI 핵심: 노이즈 제거를 반복하는 매 스텝마다, 관측 구간(컨텍스트/목표/보너스)을
**그 스텝에 맞게 다시 노이즈를 살짝 섞은 정답값으로 재치환**합니다(순수 추론시점
imputation이 아니라, 학습 때와 동일한 절차를 추론에서도 반복 — 논문이 imputation
단독보다 이게 낫다고 보인 이유).

In [12]:

@torch.no_grad()
def sample_flow(model, target, obs_mask, rel_pos, valid_mask,
                 device=DEVICE, n_steps=20, use_amp=True):
    """ODE Euler 적분. DDPM의 50스텝(DDIM 서브샘플링) 대신 10~20스텝이면 충분하다는 게
    모션 도메인 문헌(flow가 diffusion보다 훨씬 적은 스텝으로 수렴)으로 확인됨."""
    B, L, _ = target.shape
    x = torch.randn(B, L, POSE_DIM, device=device)
    dt = 1.0 / n_steps

    obs_mask_f = obs_mask.unsqueeze(-1).float()

    for i in range(n_steps):
        t_val = i * dt
        t = torch.full((B,), t_val, device=device)

        x_input = x * (1 - obs_mask_f) + target * obs_mask_f  # inpainting 유지
        flag = obs_mask.float().unsqueeze(-1)
        model_in = torch.cat([x_input, flag], dim=-1)

        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                v_pred = model(model_in, rel_pos, t, valid_mask)
        else:
            v_pred = model(model_in, rel_pos, t, valid_mask)
        v_pred = v_pred.float()

        x = x + v_pred * dt   # Euler step

    x = x * (1 - obs_mask_f) + target * obs_mask_f  # 마지막에 관측 구간 정답으로 깔끔히 덮어씀
    return x


## 11. 평가 지표 — L2Q, L2P(실제 MANO), NPSS

팀원 2가 만든 로직을 우리 row-기준 변환 + 실제 MANO로 재구현. 부호보정 쿼터니언 거리,
실제 forward kinematics, FFT 기반 리듬 유사도.


In [13]:
def l2q_error_batch(pred_90, gt_90, n_joints=N_JOINTS):
    B, T, _ = pred_90.shape
    pred_mats = rotation_6d_to_matrix_np(pred_90.reshape(B * T * n_joints, 6))
    gt_mats = rotation_6d_to_matrix_np(gt_90.reshape(B * T * n_joints, 6))
    pred_q = rotation_matrix_to_quaternion_np(pred_mats)
    gt_q = rotation_matrix_to_quaternion_np(gt_mats)
    dist = np.minimum(
        np.linalg.norm(pred_q - gt_q, axis=-1),
        np.linalg.norm(pred_q + gt_q, axis=-1),
    )
    return dist.reshape(B, T * n_joints).mean(axis=1)


def npss_batch(pred_seq, gt_seq):
    pred_fft = np.abs(np.fft.fft(pred_seq, axis=1)) ** 2
    gt_fft = np.abs(np.fft.fft(gt_seq, axis=1)) ** 2
    gt_norm = gt_fft / (gt_fft.sum(axis=1, keepdims=True) + 1e-8)
    pred_norm = pred_fft / (pred_fft.sum(axis=1, keepdims=True) + 1e-8)
    diff = np.abs(gt_norm - pred_norm).sum(axis=1)
    weight = gt_fft.sum(axis=1)
    weight = weight / (weight.sum(axis=1, keepdims=True) + 1e-8)
    return (diff * weight).sum(axis=1)


@torch.no_grad()
def l2p_error_mano_batch(pred_90, gt_90, mano_layer, n_joints=N_JOINTS):
    B, T, _ = pred_90.shape
    pred_aa = sixd_sequence_to_axis_angle(pred_90.reshape(B * T, n_joints, 6))
    gt_aa = sixd_sequence_to_axis_angle(gt_90.reshape(B * T, n_joints, 6))
    pred_joints = mano_forward(mano_layer, pred_aa).cpu().numpy()
    gt_joints = mano_forward(mano_layer, gt_aa).cpu().numpy()
    err = np.linalg.norm(pred_joints - gt_joints, axis=-1).mean(axis=-1)
    return err.reshape(B, T).mean(axis=1)

In [14]:

from scipy.spatial.transform import Rotation

def rotation_matrix_to_quaternion_np(mats):
    quats = Rotation.from_matrix(mats).as_quat()  # scipy: [x,y,z,w]
    return quats[:, [3, 0, 1, 2]]  # [w,x,y,z]로 재배열


def l2q_error(pred_90, gt_90, n_joints=N_JOINTS):
    T = pred_90.shape[0]
    pred_mats = rotation_6d_to_matrix_np(pred_90.reshape(T * n_joints, 6))
    gt_mats = rotation_6d_to_matrix_np(gt_90.reshape(T * n_joints, 6))
    pred_q = rotation_matrix_to_quaternion_np(pred_mats)
    gt_q = rotation_matrix_to_quaternion_np(gt_mats)
    dist = np.minimum(
        np.linalg.norm(pred_q - gt_q, axis=-1),
        np.linalg.norm(pred_q + gt_q, axis=-1),
    )
    return float(dist.mean())


def l2p_error_mano(pred_90, gt_90, mano_layer, n_joints=N_JOINTS):
    '''실제 MANO forward kinematics로 위치 오차 계산 (팀원들의 손수 제작 템플릿보다 정밀).'''
    T = pred_90.shape[0]
    pred_aa = sixd_sequence_to_axis_angle(pred_90.reshape(T, n_joints, 6))
    gt_aa = sixd_sequence_to_axis_angle(gt_90.reshape(T, n_joints, 6))
    pred_joints = mano_forward(mano_layer, pred_aa).cpu().numpy()
    gt_joints = mano_forward(mano_layer, gt_aa).cpu().numpy()
    return float(np.linalg.norm(pred_joints - gt_joints, axis=-1).mean())


def npss(pred_seq, gt_seq):
    pred_fft = np.abs(np.fft.fft(pred_seq, axis=0)) ** 2
    gt_fft = np.abs(np.fft.fft(gt_seq, axis=0)) ** 2
    gt_norm = gt_fft / (gt_fft.sum(axis=0, keepdims=True) + 1e-8)
    pred_norm = pred_fft / (pred_fft.sum(axis=0, keepdims=True) + 1e-8)
    diff = np.abs(gt_norm - pred_norm).sum(axis=0)
    weight = gt_fft.sum(axis=0)
    weight = weight / (weight.sum() + 1e-8)
    return float((diff * weight).sum())


def full_evaluate_diffusion(model, split="test", T_values=EVAL_T_VALUES,
                             mano_layer=mano_right, batch_size=1024, max_batches_per_T=None,
                             n_steps=20):
    model.eval()
    results = {}
    ds = DiffusionSignSparkDataset(split, mode="eval")

    for T in T_values:
        T_indices = np.where(ds.T_arr == T)[0]
        sub = torch.utils.data.Subset(ds, T_indices)
        loader = DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=collate_diffusion)

        l2q_list, l2p_list, npss_list = [], [], []
        for bi, batch in enumerate(tqdm(loader, desc=f"평가 T={T}", leave=False)):
            if max_batches_per_T is not None and bi >= max_batches_per_T:
                break
            target = batch["target"].to(DEVICE)
            obs_mask = batch["obs_mask"].to(DEVICE)
            valid_mask = batch["valid_mask"].to(DEVICE)
            rel_pos = batch["rel_pos"].to(DEVICE)

            pred = sample_flow(model, target, obs_mask, rel_pos, valid_mask, n_steps=n_steps)

            # --- 여기부터 벡터화: 배치 안 샘플을 파이썬 for문으로 하나씩 도는 대신,
            #     같은 T 안에서는 gap 위치가 전부 동일하다는 걸 이용해 배치 전체를 한 번에 처리 ---
            gap_mask = (~obs_mask) & valid_mask
            gap_idx = gap_mask[0].nonzero(as_tuple=True)[0]  # 첫 샘플 기준 -- eval에선 배치 내 전부 동일

            if len(gap_idx) == 0:
                continue

            pred_gap = pred[:, gap_idx]      # (B, T_gap, 90)
            target_gap = target[:, gap_idx]  # (B, T_gap, 90)

            pred_gap_np = pred_gap.cpu().numpy()
            target_gap_np = target_gap.cpu().numpy()

            l2q_list.extend(l2q_error_batch(pred_gap_np, target_gap_np).tolist())
            l2p_list.extend(l2p_error_mano_batch(pred_gap_np, target_gap_np, mano_layer).tolist())
            npss_list.extend(npss_batch(pred_gap_np, target_gap_np).tolist())
            # --- 벡터화 끝 ---

        results[T] = {"L2Q": np.mean(l2q_list), "L2P": np.mean(l2p_list), "NPSS": np.mean(npss_list),
                      "n": len(l2q_list)}
        print(f"T={T}: L2Q={results[T]['L2Q']:.4f} L2P={results[T]['L2P']:.4f} "
              f"NPSS={results[T]['NPSS']:.4f} (n={results[T]['n']})")

    return results


## 12. 실행

In [15]:

model = train_diffusion_silk(steps=127560, batch_size=64)


/tmp/ipykernel_558/2744069172.py:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


step       1/127560  loss=1.2333  grad_norm=2.364  lr=1.20e-05
step    2000/127560  loss=0.5955  grad_norm=0.322  lr=1.39e-05
step    4000/127560  loss=0.4922  grad_norm=0.350  lr=1.97e-05
step    6000/127560  loss=0.4667  grad_norm=0.329  lr=2.91e-05
step    8000/127560  loss=0.4536  grad_norm=0.340  lr=4.20e-05
step   10000/127560  loss=0.4458  grad_norm=0.326  lr=5.79e-05
   -> [dev, step 10000] loss: 0.4194
   -> best 갱신, 저장됨
step   12000/127560  loss=0.4418  grad_norm=0.344  lr=7.64e-05
step   14000/127560  loss=0.4385  grad_norm=0.420  lr=9.71e-05
step   16000/127560  loss=0.4381  grad_norm=0.373  lr=1.19e-04
step   18000/127560  loss=0.4345  grad_norm=0.415  lr=1.43e-04
step   20000/127560  loss=0.4337  grad_norm=0.665  lr=1.66e-04
   -> [dev, step 20000] loss: 0.4118
   -> best 갱신, 저장됨
step   22000/127560  loss=0.4337  grad_norm=0.614  lr=1.90e-04
step   24000/127560  loss=0.4305  grad_norm=0.545  lr=2.12e-04
step   26000/127560  loss=0.4292  grad_norm=0.771  lr=2.33e-04
step  

In [18]:
model = DiffusionSILKHand().to(DEVICE)
ckpt = torch.load("/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST/diffusion_silk_flow_matched_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

print(f"불러온 체크포인트 best_val: {ckpt['best_val']:.4f}")

/tmp/ipykernel_558/2744069172.py:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


불러온 체크포인트 best_val: 0.4084


In [19]:

test_results = full_evaluate_diffusion(model, split="test", batch_size=1024, n_steps=20)


평가 T=5:   0%|          | 0/145 [00:00<?, ?it/s]

T=5: L2Q=0.0306 L2P=0.0017 NPSS=0.0072 (n=147748)


평가 T=10:   0%|          | 0/141 [00:00<?, ?it/s]

T=10: L2Q=0.0569 L2P=0.0032 NPSS=0.0276 (n=143382)


평가 T=20:   0%|          | 0/132 [00:00<?, ?it/s]

T=20: L2Q=0.0959 L2P=0.0056 NPSS=0.0719 (n=134664)


평가 T=30:   0%|          | 0/124 [00:00<?, ?it/s]

T=30: L2Q=0.1206 L2P=0.0070 NPSS=0.1055 (n=126206)


In [20]:
from google.colab import runtime
runtime.unassign()